## 中级约束设计
1. 测试延迟约束，往Emp和Dept中插入互相参照的两行。PG是支持延迟约束的，MySQL不支持延迟约束，可以通过设置约束是否有效来完成
2. 将salary划分为5个区间，每个区间对应一个level值，保证每个员工的工资值和他的level值是正确对应的，这属于行级约束
3. 编写函数，输入员工的员工号，输出一个包含员工各方面信息的编码字符串，也即我们第二章中提到的智能码。比如00010002199903020002，对应编码信息如下：

0001 | 0002 | 1999 | 03 | 02 |0002
---------|----------|---------|---------|---------|---------
 员工号 | 部门号 | 出生年份 | 级别编码 | 职位编码 | 部门领导号

同学们在实现时，规范的做法是构造一张编码对照表，而不是把编码对应信息直接放在代码里面

In [4]:
def init_database():
    conn = sqlite3.connect('university.db')
    cursor = conn.cursor()
    
    # Enable foreign key support
    cursor.execute("PRAGMA foreign_keys = ON")
    
    # Create Emp table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS emp (
        eno TEXT(4) PRIMARY KEY,
        ename TEXT(50),
        birthday DATE,
        level INTEGER DEFAULT 3 CHECK(level BETWEEN 1 AND 5),
        position TEXT(10) CHECK(position IN ('教师', '教务', '会计', '秘书')),
        salary REAL CHECK(salary BETWEEN 2000 AND 200000),
        dno TEXT(4),
        FOREIGN KEY (dno) REFERENCES dept(dno) DEFERRABLE INITIALLY DEFERRED
    )
    ''')
    
    # Create Dept table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS dept (
        dno TEXT(4) PRIMARY KEY,
        dname TEXT(20) CHECK(dname IN ('数学学院', '计算机学院', '智能学院', '电子学院', '元培学院')),
        budget REAL,
        manager TEXT(4),
        FOREIGN KEY (manager) REFERENCES emp(eno) DEFERRABLE INITIALLY DEFERRED
    )
    ''')
    
    # Create code mapping table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS code_mapping (
        category TEXT NOT NULL,
        code TEXT NOT NULL,
        value TEXT NOT NULL,
        PRIMARY KEY (category, code)
    )
    ''')
    
    # Insert code mappings
    mappings = [
        # Department codes
        ('dept', '数学学院', '01'),
        ('dept', '计算机学院', '02'),
        ('dept', '智能学院', '03'),
        ('dept', '电子学院', '04'),
        ('dept', '元培学院', '05'),
        
        # Position codes
        ('position', '教师', '01'),
        ('position', '教务', '02'),
        ('position', '会计', '03'),
        ('position', '秘书', '04'),
        
        # Level codes (same as value)
        ('level', '1', '1'),
        ('level', '2', '2'),
        ('level', '3', '3'),
        ('level', '4', '4'),
        ('level', '5', '5'),
    ]
    
    cursor.executemany(
        "INSERT OR IGNORE INTO code_mapping VALUES (?, ?, ?)",
        mappings
    )
    
    conn.commit()
    return conn


In [7]:
def insert_with_circular_reference(conn):
    cursor = conn.cursor()
    
    try:
        # Start a transaction
        cursor.execute("BEGIN TRANSACTION")
        
        # Insert department first (manager will be set later)
        cursor.execute(
            "INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)",
            ('D001', '计算机学院', 5000000, None)
        )
        
        # Insert employee referencing the department
        cursor.execute(
            "INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)",
            ('E001', '张教授', date(1975, 5, 15), 4, '教师', 80000, 'D001')
        )
        
        # Now update the department to reference the employee as manager
        cursor.execute(
            "UPDATE dept SET manager = ? WHERE dno = ?",
            ('E001', 'D001')
        )
        
        conn.commit()
        print("Successfully inserted circular references")
        
    except sqlite3.Error as e:
        conn.rollback()
        print("Failed to insert circular references:", e)

def test_constraints(conn):
    cursor = conn.cursor()
    
    print("\nTesting valid inserts:")
    try:
        # Valid employee
        cursor.execute(
            "INSERT INTO emp (eno, ename, level, position, salary) VALUES (?, ?, ?, ?, ?)",
            ('E002', '李老师', 3, '教师', 50000)
        )
        print("- Valid employee inserted")
        
        # Valid department
        cursor.execute(
            "INSERT INTO dept (dno, dname, budget) VALUES (?, ?, ?)",
            ('D002', '数学学院', 3000000)
        )
        print("- Valid department inserted")
        
    except sqlite3.Error as e:
        print("Valid insert failed:", e)
    
    print("\nTesting invalid inserts:")
    
    # Invalid level
    try:
        cursor.execute(
            "INSERT INTO emp (eno, ename, level, position, salary) VALUES (?, ?, ?, ?, ?)",
            ('E003', 'Invalid', 6, '教师', 50000)
        )
    except sqlite3.IntegrityError as e:
        print("- Caught invalid level (6):", e)
    
    # Invalid position
    try:
        cursor.execute(
            "INSERT INTO emp (eno, ename, level, position, salary) VALUES (?, ?, ?, ?, ?)",
            ('E004', 'Invalid', 3, '校长', 50000)
        )
    except sqlite3.IntegrityError as e:
        print("- Caught invalid position (校长):", e)
    
    # Invalid salary
    try:
        cursor.execute(
            "INSERT INTO emp (eno, ename, level, position, salary) VALUES (?, ?, ?, ?, ?)",
            ('E005', 'Invalid', 3, '教师', 1000)
        )
    except sqlite3.IntegrityError as e:
        print("- Caught invalid salary (1000):", e)
    
    # Invalid department name
    try:
        cursor.execute(
            "INSERT INTO dept (dno, dname, budget) VALUES (?, ?, ?)",
            ('D003', '物理学院', 2000000)
        )
    except sqlite3.IntegrityError as e:
        print("- Caught invalid department name (物理学院):", e)
    
    conn.rollback()  # Rollback all test data

def generate_smart_code(conn, eno):
    """Generate a smart code for an employee based on their information"""
    cursor = conn.cursor()
    
    # Get employee information
    cursor.execute('''
    SELECT e.eno, e.ename, e.birthday, e.level, e.position, e.salary, e.dno, d.dname
    FROM emp e LEFT JOIN dept d ON e.dno = d.dno
    WHERE e.eno = ?
    ''', (eno,))
    
    emp = cursor.fetchone()
    if not emp:
        return None
    
    # Get code mappings
    def get_mapping(category, value):
        cursor.execute(
            "SELECT code FROM code_mapping WHERE category = ? AND value = ?",
            (category, str(value)))
        result = cursor.fetchone()
        return result[0] if result else '00'
    
    # Build smart code parts
    parts = []
    
    # 1-4: Employee ID (padded to 4 digits)
    parts.append(emp[0].zfill(4))
    
    # 5-6: Department code
    dept_code = get_mapping('dept', emp[7]) if emp[7] else '00'
    parts.append(dept_code)
    
    # 7-10: Birth year
    birth_year = emp[2][:4] if emp[2] else '0000'
    parts.append(birth_year)
    
    # 11-12: Position code
    pos_code = get_mapping('position', emp[4]) if emp[4] else '00'
    parts.append(pos_code)
    
    # 13: Level code
    level_code = get_mapping('level', emp[3]) if emp[3] else '0'
    parts.append(level_code)
    
    # 14-17: Salary grade (salary / 10000)
    salary_grade = str(int(emp[5] // 10000)).zfill(4) if emp[5] else '0000'
    parts.append(salary_grade)
    
    # Combine all parts
    return ''.join(parts)

def test_smart_code(conn):
    """Test the smart code generation with sample data"""
    cursor = conn.cursor()
    
    # Insert test data
    test_data = [
        # eno, ename, birthday, level, position, salary, dno
        ('T001', '王教授', '1980-08-20', 5, '教师', 150000, 'D001'),
        ('A001', '李会计', '1990-03-15', 3, '会计', 50000, 'D002'),
        ('S001', '张秘书', '1995-11-05', 2, '秘书', 30000, None),
    ]
    
    dept_data = [
        # dno, dname, budget, manager
        ('D001', '计算机学院', 5000000, 'T001'),
        ('D002', '数学学院', 3000000, None),
    ]
    
    cursor.executemany(
        "INSERT OR IGNORE INTO emp VALUES (?, ?, ?, ?, ?, ?, ?)",
        test_data
    )
    
    cursor.executemany(
        "INSERT OR IGNORE INTO dept VALUES (?, ?, ?, ?)",
        dept_data
    )
    
    conn.commit()
    
    # Generate and display smart codes
    print("\nGenerated Smart Codes:")
    for eno in ['T001', 'A001', 'S001']:
        code = generate_smart_code(conn, eno)
        print(f"{eno}: {code}")
        
        # Decode the smart code
        if code:
            print(f"  Decoded: ID={code[:4]}, Dept={code[4:6]}, Birth={code[6:10]}, "
                  f"Position={code[10:12]}, Level={code[12]}, SalaryGrade={code[13:]}")


def test():
    conn = init_database()
    
    # Test circular reference insertion
    print("\n=== Testing Circular Reference ===")
    insert_with_circular_reference(conn)
    
    # Test constraints
    print("\n=== Testing Constraints ===")
    test_constraints(conn)
    
    # Verify the circular reference was successful
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM emp WHERE eno = 'E001'")
    emp = cursor.fetchone()
    print("\nEmployee E001:", emp)
    
    cursor.execute("SELECT * FROM dept WHERE dno = 'D001'")
    dept = cursor.fetchone()
    print("Department D001:", dept)

    test_smart_code(conn)
    
    conn.close()


In [8]:
test()


=== Testing Circular Reference ===
Failed to insert circular references: UNIQUE constraint failed: dept.dno

=== Testing Constraints ===

Testing valid inserts:
- Valid employee inserted
- Valid department inserted

Testing invalid inserts:
- Caught invalid level (6): CHECK constraint failed: level BETWEEN 1 AND 5
- Caught invalid position (校长): CHECK constraint failed: position IN ('教师', '教务', '会计', '秘书')
- Caught invalid salary (1000): CHECK constraint failed: salary BETWEEN 2000 AND 200000
- Caught invalid department name (物理学院): CHECK constraint failed: dname IN ('数学学院', '计算机学院', '智能学院', '电子学院', '元培学院')

Employee E001: ('E001', '张教授', '1975-05-15', 4, '教师', 80000.0, 'D001')
Department D001: ('D001', '计算机学院', 5000000.0, 'E001')

Generated Smart Codes:
T001: T0010019800050015
  Decoded: ID=T001, Dept=00, Birth=1980, Position=00, Level=5, SalaryGrade=0015
A001: A0010019900030005
  Decoded: ID=A001, Dept=00, Birth=1990, Position=00, Level=3, SalaryGrade=0005
S001: S0010019950020003
  